# 09 — Spectral skill attribution

This notebook addresses the central research question:

\[
\boxed{\text{If FNO beats persistence, where does the gain come from?}}
\]

We decompose forecast **error energy** into predefined normalized radial-frequency bands using the same mask and Hann taper for every model.

The goal is comparative scale attribution. Because land masking and tapering change spectral amplitudes, do not interpret these values as a pure physical ocean spectrum.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from oisst_fno.metrics import daily_rmse, spectral_error_energy_by_band

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
prediction_path = ROOT / "artifacts" / "predictions" / "test_predictions.npz"
metrics_path = ROOT / "artifacts" / "metrics" / "test_metrics.json"

if not prediction_path.exists() or not metrics_path.exists():
    raise FileNotFoundError("Run notebook 08 before spectral attribution.")

bundle = np.load(prediction_path)
evaluation = json.loads(metrics_path.read_text(encoding="utf-8"))

dates = pd.to_datetime(bundle["target_dates"].astype(str))
truth = bundle["truth"]
mask = bundle["mask"].astype(bool)

## Predefined spatial-frequency bands

Radial frequency is normalized to the corner Nyquist radius. The bands are fixed **before** examining test results:

- low: \([0,0.15)\)
- lower-mid: \([0.15,0.35)\)
- upper-mid: \([0.35,0.65)\)
- high: \([0.65,1.01)\)

These are diagnostic bands, not claims about specific physical ocean length scales. A later research extension can convert grid frequencies to approximate wavelengths and repeat the analysis on fully oceanic subdomains.

In [ ]:
BAND_EDGES = (0.0, 0.15, 0.35, 0.65, 1.01)
BAND_NAMES = ("low", "lower_mid", "upper_mid", "high")

spectral_rows = []
reference_energy = spectral_error_energy_by_band(
    bundle["persistence"], truth, mask, band_edges=BAND_EDGES
)

for model_name in ("persistence", "eof_ridge", "linear_trend", "fno"):
    energy = spectral_error_energy_by_band(bundle[model_name], truth, mask, band_edges=BAND_EDGES)
    for band_name, value, reference in zip(BAND_NAMES, energy, reference_energy):
        spectral_rows.append(
            {
                "model": model_name,
                "band": band_name,
                "error_energy": float(value),
                "skill_vs_persistence_energy": float(1.0 - value / reference),
            }
        )

spectral = pd.DataFrame(spectral_rows)
spectral

In [ ]:
pivot = spectral.pivot(index="band", columns="model", values="skill_vs_persistence_energy")
pivot = pivot.reindex(BAND_NAMES)
columns = [column for column in ("eof_ridge", "linear_trend", "fno") if column in pivot]
ax = pivot[columns].plot(kind="bar", figsize=(9, 5))
ax.axhline(0.0, linewidth=1)
ax.set_ylabel("1 - error energy / persistence error energy")
ax.set_title("Scale-resolved skill relative to persistence")
plt.tight_layout()
plt.show()

## Season and validation-defined forecast difficulty

Season is predefined.

The “hard persistence” threshold is **not** chosen from the test period. Notebook 05 defines it as the 75th percentile of validation persistence daily RMSE, and notebook 08 carries that frozen threshold into this analysis.

In [ ]:
fno_daily = daily_rmse(bundle["fno"], truth, mask)
persistence_daily = bundle["persistence_daily_rmse"].astype(float)
eof_daily = daily_rmse(bundle["eof_ridge"], truth, mask)

season_lookup = {
    12: "DJF", 1: "DJF", 2: "DJF",
    3: "MAM", 4: "MAM", 5: "MAM",
    6: "JJA", 7: "JJA", 8: "JJA",
    9: "SON", 10: "SON", 11: "SON",
}
season = np.array([season_lookup[month] for month in dates.month])

hard_threshold = float(evaluation["validation_defined_hard_persistence_threshold_c"])
hard_case = persistence_daily > hard_threshold

case_frame = pd.DataFrame(
    {
        "date": dates,
        "season": season,
        "hard_by_validation_threshold": hard_case,
        "persistence_rmse_c": persistence_daily,
        "fno_rmse_c": fno_daily,
        "eof_ridge_rmse_c": eof_daily,
        "fno_minus_persistence_c": fno_daily - persistence_daily,
        "eof_minus_persistence_c": eof_daily - persistence_daily,
    }
)
case_frame.head()

In [ ]:
season_summary = (
    case_frame.groupby("season", observed=True)
    .agg(
        n=("date", "size"),
        persistence_rmse_c=("persistence_rmse_c", "mean"),
        fno_rmse_c=("fno_rmse_c", "mean"),
        eof_ridge_rmse_c=("eof_ridge_rmse_c", "mean"),
        fno_minus_persistence_c=("fno_minus_persistence_c", "mean"),
    )
    .reindex(["DJF", "MAM", "JJA", "SON"])
)

regime_summary = case_frame.groupby("hard_by_validation_threshold", observed=True).agg(
    n=("date", "size"),
    persistence_rmse_c=("persistence_rmse_c", "mean"),
    fno_rmse_c=("fno_rmse_c", "mean"),
    eof_ridge_rmse_c=("eof_ridge_rmse_c", "mean"),
    fno_minus_persistence_c=("fno_minus_persistence_c", "mean"),
)

display(season_summary)
display(regime_summary)

In [ ]:
metrics_dir = ROOT / "artifacts" / "metrics"
spectral.to_csv(metrics_dir / "spectral_skill_by_band.csv", index=False)
case_frame.to_csv(metrics_dir / "test_case_errors.csv", index=False)
season_summary.to_csv(metrics_dir / "season_skill_summary.csv")
regime_summary.to_csv(metrics_dir / "persistence_difficulty_summary.csv")

## Questions this notebook must answer

1. Is FNO skill positive in low, middle, or high spatial-frequency bands?
2. Does EOF+ridge capture the same bands?
3. Is aggregate FNO skill actually a low-frequency effect?
4. Is high-frequency structure degraded even when RMSE improves?
5. Are gains concentrated in one season?
6. Does FNO help more when persistence is genuinely challenged, using the threshold frozen on validation?

A useful research result is not simply “FNO wins.” A more informative conclusion might be:

> FNO adds little on low-frequency slowly varying SST, but reduces mid-frequency error on validation-defined difficult forecast cases.

Or the opposite. The data decide.

### Statistical caution

Season and regime tables here are primarily **descriptive attribution**. Do not attach naive IID confidence intervals to them. For paper-level subgroup inference, use a temporally valid resampling design and prespecify the contrasts.